In [3]:


# Import standard libraries for file handling and text processing
import os, pathlib, textwrap, glob

# Load documents from various sources (URLs, text files, PDFs)
from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader

# Split long texts into smaller, manageable chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Vector store to store and retrieve embeddings efficiently using FAISS
from langchain.vectorstores import FAISS

# Generate text embeddings using OpenAI or Hugging Face models
from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings, SentenceTransformerEmbeddings

# Use local LLMs (e.g., via Ollama) for response generation
from langchain.llms import Ollama

# Build a retrieval chain that combines a retriever, a prompt, and an LLM
from langchain.chains import ConversationalRetrievalChain

# Create prompts for the RAG system
from langchain.prompts import PromptTemplate

print("✅ Libraries imported! You're good to go!")



/opt/miniconda3/envs/rag-chatbot/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Libraries imported! You're good to go!


1 - Data preparation

The goal of this step is to turn all reference documents into small chunks of text that a retriever can index and search. These documents typically come from:

    PDF files: local documents such as policies, user manuals, or guides.
    Web pages (HTML): online documentation, blog posts, or help articles.

In this step, we perform two actions:

    Ingesting: load every PDF and collect the raw text in a list named raw_docs.
    Chunking: split each document into small, overlapping chunks so later steps can match a user query to the most relevant passage.

    Step 1. Load the pdfs and get the data
    Step 2. Data chunking
    Step 3. Embedding creation of chunks and indexing




2.1 - Ingest source documents

We can use different libraries to load and process PDFs. A quick web search will show several options, each with its own strengths. In this case, we’ll use PyPDFLoader from LangChain, which makes it easy to extract text from PDF files for downstream processing. To learn more about how to use it, refer to: https://python.langchain.com/docs/integrations/document_loaders/pypdfloader/

Use PyPDFLoader to load every PDF whose filename matches data/Everstorm_*.pdf and collect all pages in a list called raw_docs. The content of these PDFs is synthetically generated for educational purposes.


In [6]:
pdf_paths = glob.glob("data/Everstorm_*.pdf")
raw_docs = []

for file_path in pdf_paths:
    try:
        loader = PyPDFLoader(file_path)
        docs = loader.load()

        # adding source to all doc in docs
        if docs:
            for doc in docs:
                doc.metadata['source'] = file_path
            raw_docs.extend(docs)
    
    except Exception as e:
        print(f"Error loading documents {file_path} : {e}")
        continue

print(f"Loaded {len(raw_docs)} PDF pages from {len(pdf_paths)} files.")
print(raw_docs)

Ignoring wrong pointing object 81 0 (offset 0)
Ignoring wrong pointing object 76 0 (offset 0)
Ignoring wrong pointing object 80 0 (offset 0)


Loaded 8 PDF pages from 4 files.
[Document(metadata={'producer': 'Skia/PDF m138 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Return_and_exchange_policy', 'source': 'data/Everstorm_Return_and_exchange_policy.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Everstorm  Outfitters    RETURN  &  EXCHANGE  POLICY    Document  ROX-2025-05   Easy-Fit  Promise    If  your  gear  doesn’t  fit  or  just  isn’t  your  vibe,  send  it  back  within  **30  days**  of  delivery  for  a  refund  or  free  size  exchange.   Eligibility  Checklist    ●  Unworn,  unwashed,  no  odors,  tags  attached    ●  Original  shoe  box  (footwear)  placed  inside  outer  carton    ●  Electronics  (power-banks,  headlamps)  unopened  unless  faulty   How  to  Start    ●  Visit  everstorm.example/returns  →  enter  order  #  and  email.    ●  Select  “Refund”  or  “Exchange.”    ●  Print  prepaid  label;  pack  securely.  Multiple  items  can  share  one  box.   Instan

Customer‑Support Chatbot for an E-Commerce Store